# Process PA DMS data from Chen et al.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from collections import defaultdict
from Bio import SeqIO
import subprocess

In [ ]:
data_dir = "../../flu-usher/results"  # default for interactive use


In [ ]:
# Parameters
data_dir = "/fh/fast/matsen_e/hhaddox/2025/flu-usher/results"


Parse the AA mutation column to extract wildtype AA, DMS site, and mutant AA. Exclude indel (`_`) and stop codon (`*`) mutations. Average the two no-drug fitness replicates (P1NO), group duplicate nucleotide mutations that produce the same AA change, and compute `dms_effect = log(mean fitness)`. Then align the DMS protein sequence (first 240 AA of PA) to the PA tree reference using MUSCLE to establish site numbering correspondence.

In [ ]:
# Read reference PA protein from flu-usher
ref_fasta = os.path.join(data_dir, 'PA/all/curated_reference.fasta')
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate()).replace('*', '')
print('Reference PA sequence length:', len(ref_aa_seq))

# Read DMS data from Excel
raw_dms = pd.read_excel('../data/dms_data/Chen_PA/fitness calculation.xlsx', sheet_name='HcMut')

# Parse AA mutation column: extract wt_aa, dms_site, mut_aa
# Exclude '_' (indel) and '*' (stop codon) mutations; keep only standard AA substitutions
valid_aa_pat = re.compile(r'^([A-Z])(\d+)([A-Z])$')
raw_dms['parsed'] = raw_dms['AA'].apply(lambda x: valid_aa_pat.match(str(x)))
raw_dms = raw_dms[raw_dms['parsed'].notna()].copy()
raw_dms['wt_aa']    = raw_dms['parsed'].apply(lambda m: m.group(1))
raw_dms['dms_site'] = raw_dms['parsed'].apply(lambda m: int(m.group(2)))
raw_dms['mut_aa']   = raw_dms['parsed'].apply(lambda m: m.group(3))

# Compute per-row mean no-drug fitness, then group by AA mutation and average
# (multiple nucleotide changes can produce the same AA change)
raw_dms['fitness'] = (raw_dms['P1NO-1-fit'] + raw_dms['P1NO-2-fit']) / 2.0
pa_dms_data = (
    raw_dms
    .groupby(['dms_site', 'wt_aa', 'mut_aa'], as_index=False)['fitness']
    .mean()
)
pa_dms_data['dms_effect'] = np.log(pa_dms_data['fitness'])

# Exclude synonymous AA mutations
pa_dms_data = pa_dms_data[pa_dms_data['wt_aa'] != pa_dms_data['mut_aa']].copy()

print('DMS site range:', pa_dms_data['dms_site'].min(), '-', pa_dms_data['dms_site'].max())

# Build DMS protein sequence from wt_aa at each site (sites 1-240)
dms_seq_df = (
    pa_dms_data
    .sort_values('dms_site')
    .drop_duplicates('dms_site')[['dms_site', 'wt_aa']]
)
dms_seq_df = dms_seq_df[dms_seq_df['dms_site'].between(1, 240)]
aa_seq = ''.join(dms_seq_df.set_index('dms_site')['wt_aa'].reindex(range(1, 241)).dropna())
print('DMS sequence length:', len(aa_seq))

# Save sequences for MUSCLE alignment
output_dir = '../results/dms_data/Chen_PA/'
os.makedirs(output_dir, exist_ok=True)
unaligned_fasta = os.path.join(output_dir, 'unaligned.fasta')
if not os.path.isfile(unaligned_fasta):
    with open(unaligned_fasta, 'w') as f:
        f.write('>reference\n' + ref_aa_seq + '\n')
        f.write('>dms\n'       + aa_seq       + '\n')

# Align the sequences using MUSCLE
aligned_fasta = os.path.join(output_dir, 'aligned.fasta')
if not os.path.isfile(aligned_fasta):
    cmd = ['muscle', '-align', unaligned_fasta, '-output', aligned_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

In [ ]:
# Read in the aligned sequences
seqs_dict = {}
aligned_records = list(SeqIO.parse(aligned_fasta, 'fasta'))
for record in aligned_records:
    seqs_dict[record.id] = str(record.seq)

aligned_ref_seq = seqs_dict['reference']
aligned_dms_seq = seqs_dict['dms']

# Compute percent identity
n_sites_to_compare = 0
n_identical = 0
for (ref_aa, dms_aa) in zip(aligned_ref_seq, aligned_dms_seq):
    if ref_aa != '-' and dms_aa != '-':
        n_sites_to_compare += 1
        if ref_aa == dms_aa:
            n_identical += 1

print(n_sites_to_compare, n_identical, n_identical / n_sites_to_compare)

# Determine numbering scheme
dms_start_index = int(dms_seq_df['dms_site'].min())
numbering_dict = defaultdict(list)
assert len(aligned_ref_seq) == len(aligned_dms_seq)
for (seq_id, seq, start_index) in [
    ('tree_reference_site', aligned_ref_seq, 1),
    ('dms_site', aligned_dms_seq, dms_start_index),
]:
    seq_index = start_index
    for (alignment_index, aa) in enumerate(seq, 1):
        if aa != '-':
            numbering_dict['alignment_index'].append(alignment_index)
            numbering_dict['seq_id'].append(seq_id)
            numbering_dict['seq_index'].append(seq_index)
            numbering_dict['seq_aa'].append(aa)
            seq_index += 1

alignment_numbering_df = (
    pd.DataFrame(numbering_dict)
    .pivot(index='alignment_index', columns='seq_id', values='seq_index')
    .reset_index()
    .rename_axis(None, axis=1)
    .dropna()
    [['dms_site', 'tree_reference_site']]
)
alignment_numbering_df['dms_site'] = alignment_numbering_df['dms_site'].astype(int)
alignment_numbering_df['tree_reference_site'] = alignment_numbering_df['tree_reference_site'].astype(int)
alignment_numbering_df.head()

In [ ]:
# Merge DMS data with numbering scheme and save
pa_dms_data_processed = (
    pa_dms_data
    .merge(alignment_numbering_df, on='dms_site', validate='many_to_one')
)

pa_dms_data_processed = pa_dms_data_processed[['dms_site', 'wt_aa', 'mut_aa', 'tree_reference_site', 'dms_effect']]
pa_dms_data_processed.to_csv(os.path.join(output_dir, 'processed_dms_data.csv'), index=False)
print('Number of mutations with processed data:', len(pa_dms_data_processed))
pa_dms_data_processed.head()

In [ ]:
pa_dms_data_processed.tail()

In [ ]:
sum(pa_dms_data_processed['dms_site'] != pa_dms_data_processed['tree_reference_site'])